# <font color='green'><b><u>Single-cell downstream report<u></b></font>

In [ ]:
import scanpy as sc

In [ ]:
path_adata = FILE

In [ ]:
adata = sc.read(path_adata)

In [ ]:
from IPython.display import display, HTML

display(HTML("<p>After filtering, the data set contains " + str(adata.n_obs) + " cells and " + str(adata.n_vars) 
             + " genes or transcripts. The number of cells per sample are listed below:</p>"))

In [ ]:
adata.obs.value_counts('sample').rename_axis('samples').reset_index(name='number of cells')

## <font color='green'>Highly variable genes</font>

In [ ]:
sc.pl.highly_variable_genes(adata)

## <font color='green'>PCA</font>

Sample distribution

In [ ]:
sc.pl.pca(
    adata,
    color="sample",
    dimensions=[(0, 1)],
)

QC metrics

In [ ]:
sc.pl.pca(
    adata,
    color=["total_counts", "pct_counts_mt", "pct_counts_ribo"],
    dimensions=[(0, 1)],
)

Loadings indicate the most informative genes across the first PCs

In [ ]:
n_pcs = 6
n_pcs =  adata.obsm['X_pca'].shape[1] if n_pcs > adata.obsm['X_pca'].shape[1] else n_pcs
list_pcs = list(range(1, n_pcs+1))
sc.pl.pca_loadings(adata, include_lowest=False, components=list_pcs)

---
## <font color='green'>UMAP</font>

In [ ]:
adata.obsm['X_pca_umap'] =  adata.obsm['X_umap'].copy()

Samples distribution

In [ ]:
sc.pl.umap(adata, color=['sample'])

In [ ]:
if 'X_umap_scvi' in adata.obsm:
    sc.pl.embedding(adata, basis='X_umap_scvi', color=['sample'], title='scvi UMAP')

QC metrics

In [ ]:
sc.pl.umap(adata, color=['n_genes_by_counts', 'pct_counts_mt'])

In [ ]:
sc.pl.umap(adata, color=['pct_counts_ribo', 'total_counts_hb'])

Distribution of predicted doublets

In [ ]:
sc.pl.umap(adata, color=['predicted_doublet', 'doublet_score'])

---
## <font color='green'>Predicted labels</font>

In [ ]:
sc.pl.umap(adata, color=['label'])

---
## <font color='green'>Clusters</font>

In [ ]:
clustering_label = 'leiden'

In [ ]:
# define order of clusters numerically
import numpy as np

list_clusters = np.unique(adata.obs[clustering_label])
list_clusters = sorted([int(x) for x in list_clusters])
list_clusters = [str(x) for x in list_clusters]

In [ ]:
sc.pl.umap(adata, color=[clustering_label])

In [ ]:
if 'X_umap_scvi' in adata.obsm:
    sc.pl.embedding(adata, basis='X_umap_scvi', color=[clustering_label], title='scvi UMAP')

---
## <font color='green'><b>Marker genes</b></font>

Marker genes per clusters are identified based on a Wilcoxon rank-sum test by comparing the gene expression between cells of a cluster against all other cells.

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, groupby="leiden", standard_scale="var", swap_axes=True, n_genes=3)

In [ ]:
import numpy as np
from IPython.display import display, HTML


for clus in list_clusters:
    display(HTML("<h3><font color='green'>Markers cluster " + clus + 
                 "</font></h3><details><div style='overflow-y: scroll; max-height: 400px'>" + 
             sc.get.rank_genes_groups_df(adata, group=clus).head(100).to_html() + "</div></details>"))

---
## <font color='green'><b>Gene Set Enrichment</b></font>

g:Profiler gene set enrichment per cluster. Cluster marker genes are further filtered by their expression inside and outside of the cluster before computing enrichment.

In [ ]:
import numpy as np
from IPython.display import display, HTML


for clus in list_clusters:
    display(HTML("<h3><font color='green'>Enrichment cluster " + clus + 
                 "</font></h3><details><div style='overflow-y: scroll; overflow-x: scroll; max-height: 400px'>" + 
             adata.uns['rank_genes_groups']['enrich'][clus].head(100).to_html() + "</div></details>"))

---

## <font color='green'><b>Cell-cell interaction</b></font>

LIANA+ consensus ligand-receptor interactions. LIANA+ uses ranking and aggregating to compute a `magnitude rank`. A lower `magnitude rank` indicates that an interaction is more likely to occur.

Following methods are currently implemented in the LIANA+ framework:

In [ ]:
import liana as li

li.mt.show_methods()

In [ ]:
# create images
import numpy as np
import matplotlib.pyplot as plt
import math

ntop = 20

paths_images = {}

img_width = int(math.ceil(len(list_clusters) / 2))
img_hieght = int(math.ceil(ntop / 2))

if 'liana_res' in adata.uns:
    for clus in list_clusters:
        
        path_img = f"./dotplot_test_clus{clus}.png"
        paths_images[clus] = path_img
                        
        fig = li.pl.dotplot(adata = adata, 
              colour='magnitude_rank',
              size='specificity_rank',
              inverse_size=True,
              inverse_colour=True,
              source_labels=[clus],
              target_labels=list_clusters,
              top_n=ntop, 
              orderby='magnitude_rank',
              orderby_ascending=True,
              figure_size=(img_width, img_hieght),
              return_fig=True)

        fig.save(path_img)

In [ ]:
# HTML output
from IPython.display import display, HTML
import base64

if 'liana_res' in adata.uns:
    df_liana = adata.uns['liana_res'].copy()
    
    for clus, path in paths_images.items():
        with open(path, "rb") as f_plot:
            # dotplot
            image_string = base64.b64encode(f_plot.read()).decode("utf-8")
            image_html = f'<h3><font color="green">Interactions cluster {clus}</h3></font>'
            image_html += '<details><summary>Click to expand dotplot</summary>'
            image_html += f'<figure><img alt="liana image" src="data:image/png;base64,{image_string}" />'
            image_html += f'<figcaption>Shown are the top {ntop} interactions between cluster {clus} and the other clusters</figcaption></figure></details>'
            
            # table
            df_liana_clus = df_liana[ df_liana['source'] == clus]
            image_html += '<font color="black"><details><summary>Click to expand table</summary>'
            image_html += "<div style='overflow-y: scroll; overflow-x: scroll; max-height: 400px'>" + df_liana_clus.head(100).to_html() + "</div></details>"
            
            display(HTML(image_html))